# Nurse Rostering (INRC-II): exato, GRASP e matheurística

Antes de rodar, envie pelo painel de arquivos (ícone de pasta, à esquerda):
* `nurse_rostering.zip` (o projeto)
* `testdatasets_json.zip` (instâncias de teste do INRC-II)
* opcional: `datasets_json.zip` (instâncias da competição, de 30 a 120 enfermeiros)

In [ ]:
!pip -q install highspy swiglpk
!unzip -q -o nurse_rostering.zip -d .
!mkdir -p dados && unzip -q -o testdatasets_json.zip -d dados
!test -f datasets_json.zip && unzip -q -o datasets_json.zip -d dados || echo "(sem datasets_json.zip)"
%cd nurse_rostering
!ls ../dados

## 1. Testes de correção
Deve terminar com `TUDO CERTO` (leva cerca de 1 minuto).

In [ ]:
!python testes.py ../dados

## 2. Rodar cada método numa instância
Argumentos: `pasta historico semanas tempo_em_segundos`.

In [ ]:
!python exato.py ../dados/n005w4 0 1,2,3,3 60

In [ ]:
!python grasp.py ../dados/n005w4 0 1,2,3,3 60

In [ ]:
!python lns.py ../dados/n021w4 0 5,4,1,2 120

## 3. Experimentos
Roda exato, GRASP e LNS nas instâncias de `instancias.txt` e grava `resultados.csv`.
Tempo total ≈ instâncias × (1 + 2 × sementes) × tempo. Com os valores abaixo: 3 × 3 × 60 s ≈ 9 min.
Para instâncias da competição, acrescentem linhas em `instancias.txt`, por exemplo `n030w4 1 6,2,9,1`.

In [ ]:
!python experimentos.py --dados ../dados --instancias instancias.txt --tempo 60 --sementes 1 --threads 1

In [ ]:
import pandas as pd
r = pd.read_csv("resultados.csv")
display(r)
tab = r.groupby(["instancia", "metodo"]).agg(custo_medio=("custo", "mean"), melhor=("custo", "min"),
                                             tempo_s=("tempo_s", "mean")).unstack("metodo")
display(tab)

In [ ]:
import json, matplotlib.pyplot as plt
conv = json.load(open("resultados_convergencia.json"))
inst = sorted({k.split("|")[0] for k in conv})[-1]          # troque pelo nome da instancia desejada
for k, h in conv.items():
    if k.startswith(inst) and h:
        t, c = zip(*h)
        plt.step(t, c, where="post", label=k.split("|")[1])
plt.xlabel("tempo (s)"); plt.ylabel("melhor custo"); plt.title(inst); plt.legend(); plt.show()

## 4. (Opcional) O mesmo modelo no AMPL
Precisa de licença: a Community Edition é gratuita em ampl.com/ce. Sem licença, o AMPL só aceita até 500 variáveis, e estas instâncias têm milhares. Cole a sua licença abaixo. Para usar o Gurobi, incluam `"gurobi"` em `modules` e troquem o solver em `rodar.run`.

In [ ]:
!pip -q install amplpy
from amplpy import ampl_notebook
ampl = ampl_notebook(modules=["highs"], license_uuid="COLE-SUA-LICENCA-AQUI")
ampl.eval("include rodar.run;")
print(open("resultados_ampl.csv").read())

## 5. Baixar os resultados

In [ ]:
from google.colab import files
files.download("resultados.csv")